In [34]:
import pandas as pd             # data package
import matplotlib.pyplot as plt # graphics 
import datetime as dt
import numpy as np
import requests
from io import StringIO

import pyarrow as pa
import pyarrow.parquet as pq

In [35]:
df = pd.read_parquet('TOTALdata-current.parquet')

df.rename(columns={'I_COMMODITY': 'HS10'}, inplace=True)

df["HS2"] = df["HS10"].str[0:2]

df["HS6"] = df["HS10"].str[0:6]

df["HS10"] = df["HS10"]

df.time = pd.to_datetime(df.time, format="%Y-%m")

df["imports"] = df["CON_VAL_MO"].astype(float)

df["duty"] = df["CAL_DUT_MO"].astype(float)

df = df.set_index("time").loc["2024"]

df = df.groupby("HS10").aggregate({"HS10": "first", "imports": "sum", "HS6":"first", "I_COMMODITY_SDESC": "first"} )

df = df.reset_index(drop=True)

In [36]:
df.head()

,HS10,imports,HS6,I_COMMODITY_SDESC
0,0101210010,59256137.0,010121,"HORSES, LIVE, PUREBRED BREEDING MALE"
1,0101210020,105458357.0,010121,"HORSES, LIVE, PUREBRED BREEDING FEMALE"
2,0101290090,652482722.0,010129,"HORSES, LIVE, NESOI"
3,0101300000,33419.0,010130,"ASSES, LIVE"
4,0102210010,147200.0,010221,"CATTLE, LIVE, PUREBRED BREEDING MALE, DAIRY"


In [37]:
import pandas as pd
import requests
from io import StringIO

# Read the concordance file from Census URL
url = "https://www.census.gov/foreign-trade/schedules/b/2022/imp-code.txt"
response = requests.get(url)
response.raise_for_status()

# Parse as fixed-width format based on imp-stru.txt specification
# Column positions (converting 1-indexed to 0-indexed for Python)
colspecs = [
    (0, 10),       # Characters 1-10: COMMODITY (HTS/HS10 code)
    (14, 65),      # Characters 15-65: Descrip_1 (Short description)
    (69, 219),     # Characters 70-219: Descrip_2 (Long description)
    (224, 227),    # Characters 225-227: QUANTITY_1 (First unit)
    (232, 235),    # Characters 233-235: QUANTITY_2 (Second unit)
    (240, 245),    # Characters 241-245: SITC (Standard International Trade Classification)
    (250, 255),    # Characters 251-255: END_USE
    (260, 261),    # Character 261: USDA (Agriculture/non-agriculture code)
    (265, 271),    # Characters 266-271: NAICS classification
    (276, 278)     # Characters 277-278: HITECH classification
]

column_names = ['HS10', 'short_desc', 'long_desc', 'quantity_1', 'quantity_2', 
                'sitc', 'end_use', 'usda', 'naics', 'hitech']

# Specify dtype for HS10 to ensure it's read as string (preserves leading zeros)
concordance_df = pd.read_fwf(StringIO(response.text), colspecs=colspecs, names=column_names, 
                              header=None, dtype={'HS10': str})

# Clean up whitespace
for col in concordance_df.columns:
    if concordance_df[col].dtype == 'object':
        concordance_df[col] = concordance_df[col].str.strip()

concordance_df["HS6"] = concordance_df["HS10"].str[0:6]

concordance_df["HS8"] = concordance_df["HS10"].str[0:8]

# Save as CSV
output_file = 'census_imp_code_2022.csv'
concordance_df.to_csv(output_file, index=False)

print(f"Concordance saved to {output_file}")
print(f"Shape: {concordance_df.shape}")
concordance_df.head()

Concordance saved to census_imp_code_2022.csv
Shape: (19758, 12)


,HS10,short_desc,long_desc,quantity_1,quantity_2,sitc,end_use,usda,naics,hitech,HS6,HS8
0,0101210010,"HORSES, LIVE, PUREBRED BREEDING MALE","HORSES, LIVE, PUREBRED BREEDING MALE",NO,NaN,00150,12060,0,112920,0,010121,01012100
1,0101210020,"HORSES, LIVE, PUREBRED BREEDING FEMALE","HORSES, LIVE, PUREBRED BREEDING FEMALE",NO,NaN,00150,12060,0,112920,0,010121,01012100
2,0101290010,"HORSES, LIVE, IMPORTED FOR IMMEDIATE SLAUGHTER","HORSES, LIVE, IMPORTED FOR IMMEDIATE SLAUGHTER",NO,NaN,00150,100,0,112920,0,010129,01012900
3,0101290090,"HORSES, LIVE, NESOI","HORSES, LIVE, NESOI",NO,NaN,00150,12060,0,112920,0,010129,01012900
4,0101300000,"ASSES, LIVE","ASSES, LIVE",NO,NaN,00150,12060,0,112920,0,010130,01013000


In [38]:
# Compare hs10_codes by merging df with concordance_df
# HS10 in df is already a string with leading zeros preserved
df_with_str = df.copy()
df_with_str['HS8'] = df_with_str['HS10'].str[0:8]

# Merge with indicator to see where matches occur
merged = df_with_str.merge(
    concordance_df[['HS10', 'naics', 'short_desc']], 
    on='HS10', 
    how='left', 
    indicator=True,
    suffixes=('', '_conc')
)

# Analyze merge results
print("=== Merge Results ===")
print(merged['_merge'].value_counts())
print(f"\nTotal rows in df: {len(df):,}")
print(f"Rows with concordance match (both): {(merged['_merge'] == 'both').sum():,}")
print(f"Rows without concordance match (left_only): {(merged['_merge'] == 'left_only').sum():,}")

# Unique HS10 codes
unique_matched = merged[merged['_merge'] == 'both']['HS10'].nunique()
unique_unmatched = merged[merged['_merge'] == 'left_only']['HS10'].nunique()
total_unique_in_df = df['HS10'].nunique()

print(f"\n=== Unique HS10 Codes ===")
print(f"Total unique HS10 in df: {total_unique_in_df:,}")
print(f"Matched in concordance: {unique_matched:,} ({unique_matched/total_unique_in_df*100:.1f}%)")
print(f"Not matched in concordance: {unique_unmatched:,} ({unique_unmatched/total_unique_in_df*100:.1f}%)")

# Trade value analysis
matched_imports = merged[merged['_merge'] == 'both']['imports'].sum()
unmatched_imports = merged[merged['_merge'] == 'left_only']['imports'].sum()
total_imports = df['imports'].sum()

print(f"\n=== Trade Value Analysis ===")
print(f"Total imports: ${total_imports/1e9:.2f}B")
print(f"Imports with concordance match: ${matched_imports/1e9:.2f}B ({matched_imports/total_imports*100:.1f}%)")
print(f"Imports without concordance match: ${unmatched_imports/1e9:.2f}B ({unmatched_imports/total_imports*100:.1f}%)")



=== Merge Results ===
_merge
both          18254
left_only       446
right_only        0
Name: count, dtype: int64

Total rows in df: 18,700
Rows with concordance match (both): 18,254
Rows without concordance match (left_only): 446

=== Unique HS10 Codes ===
Total unique HS10 in df: 18,700
Matched in concordance: 18,254 (97.6%)
Not matched in concordance: 446 (2.4%)

=== Trade Value Analysis ===
Total imports: $3251.19B
Imports with concordance match: $3101.26B (95.4%)
Imports without concordance match: $149.94B (4.6%)


In [39]:
# Top unmatched codes by import value
print(f"\nTop 10 unmatched HS10 codes by import value:")
unmatched_df = merged[merged['_merge'] == 'left_only']
top_unmatched_agg = unmatched_df.groupby('HS10').agg({
    'imports': 'sum',
    'I_COMMODITY_SDESC': 'first'
}).sort_values('imports', ascending=False).head(10)

for hs10, row in top_unmatched_agg.iterrows():
    print(f"  {hs10}: ${row['imports']/1e6:.1f}M - {row['I_COMMODITY_SDESC']}")


Top 10 unmatched HS10 codes by import value:
  8703220110: $28176.1M - PASS MOTOR VEH, SPARK IGN ENG, (1000-1500 CC), NEW
  8703800060: $18666.7M - PASS VEHICLES,ONLY ELECTR MOTOR GT 250MI NEW,NESOI
  3004909221: $12244.5M - CARDIOVASCULAR MEDICAMENTS, NESOI
  3004909239: $9546.1M - ANTIDEPRESSANTS, TRANQUILIZERS,OTHER PSYCH AGT,NES
  3004909202: $5744.8M - ANTIVIRALS, IN DOSES OR PACKAGED FOR RETAIL
  8542310045: $4689.9M - PRCSSRS (INCL MICRO): CENTRL PROCSSNG UNITS (CPUS)
  8542390060: $4094.3M - ELECTRONIC INTEGRATED CIRCUITS, NESOI
  8703800020: $4092.2M - PASSENGER VEHICLES, ONLY ELECTR MOTOR LT=250MI NEW
  8542310050: $3752.2M - PROCESSORS (INCLUDING MICROPROCESSORS): OTHER
  3926909989: $3626.7M - ARTICLES OF PLASTICS ETC, NESOI


In [40]:
# Create final dataframe: HS10, NAICS (direct match), associated_naics (from HS8), long_desc

# Step 1: Get unique HS10 codes from df (HS10 is already a string)
hs10_df = df[['HS10']].drop_duplicates().copy()
hs10_df['HS8'] = hs10_df['HS10'].str[0:8]

# Verify types
print("=== Type Check ===")
print(f"df HS10 type: {df['HS10'].dtype}")
print(f"concordance_df HS10 type: {concordance_df['HS10'].dtype}")
print(f"Sample df HS10: {hs10_df['HS10'].iloc[0]} (len={len(hs10_df['HS10'].iloc[0])})")
print(f"Sample concordance HS10: {concordance_df['HS10'].iloc[0]} (len={len(concordance_df['HS10'].iloc[0])})")
print(f"Sample df HS8: {hs10_df['HS8'].iloc[0]}")
print(f"Sample concordance HS8: {concordance_df['HS8'].iloc[0]}")

# Step 2: Merge with concordance to get direct NAICS match and long_desc
result_df = hs10_df.merge(
    concordance_df[['HS10', 'naics', 'long_desc']].drop_duplicates(),
    on='HS10',
    how='left'
)
result_df.rename(columns={'naics': 'naics_direct'}, inplace=True)

# Step 3: Create HS8 to NAICS list mapping (all NAICS codes for each HS8)
hs8_naics_list = concordance_df.groupby('HS8')['naics'].apply(
    lambda x: list(x.dropna().unique())
).reset_index()
hs8_naics_list.rename(columns={'naics': 'associated_naics'}, inplace=True)

# Step 4: Merge HS8 NAICS list
result_df = result_df.merge(
    hs8_naics_list,
    on='HS8',
    how='left'
)

# Step 5: Set associated_naics to empty list where there's a direct match
result_df['associated_naics'] = result_df.apply(
    lambda row: [] if pd.notna(row['naics_direct']) else (row['associated_naics'] if isinstance(row['associated_naics'], list) else []),
    axis=1
)

# Clean up and select final columns
result_df = result_df[['HS10', 'naics_direct', 'associated_naics', 'long_desc']].rename(
    columns={'naics_direct': 'naics'}
)

print(f"\n=== Results ===")
print(f"Final dataframe shape: {result_df.shape}")
print(f"Rows with direct NAICS match: {result_df['naics'].notna().sum():,}")
print(f"Rows with associated NAICS (from HS8): {(result_df['associated_naics'].apply(len) > 0).sum():,}")
print(f"Rows with no NAICS at all: {((result_df['naics'].isna()) & (result_df['associated_naics'].apply(len) == 0)).sum():,}")
print(f"Rows with long_desc: {result_df['long_desc'].notna().sum():,}")

result_df.head(20)

=== Type Check ===
df HS10 type: object
concordance_df HS10 type: object
Sample df HS10: 0101210010 (len=10)
Sample concordance HS10: 0101210010 (len=10)
Sample df HS8: 01012100
Sample concordance HS8: 01012100

=== Results ===
Final dataframe shape: (18700, 4)
Rows with direct NAICS match: 18,254
Rows with associated NAICS (from HS8): 446
Rows with no NAICS at all: 0
Rows with long_desc: 18,254


,HS10,naics,associated_naics,long_desc
0,0101210010,112920,[],"HORSES, LIVE, PUREBRED BREEDING MALE"
1,0101210020,112920,[],"HORSES, LIVE, PUREBRED BREEDING FEMALE"
2,0101290090,112920,[],"HORSES, LIVE, NESOI"
3,0101300000,112920,[],"ASSES, LIVE"
4,0102210010,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING MALE, DAIRY"
5,0102210020,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING FEMALE, DAIRY"
6,0102210030,11211X,[],"CATTLE, PUREBRED BREEDING, MALE, LIVE, EXCEPT ..."
7,0102210050,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING FEMALE, EXCEPT..."
8,0102292011,11211X,[],"CATTLE, IMPORTED SPECIALLY FOR DAIRY PURPOSES,..."
9,0102292012,11211X,[],"CATTLE, IMPORTED SPECIALLY FOR DAIRY PURPOSES,..."


In [41]:
# Show instances where associated_naics is active (non-empty)
active_associated = result_df[result_df['associated_naics'].apply(len) > 0].copy()

print(f"HS10 codes with associated NAICS (from HS8): {len(active_associated):,}")
print(f"\nExamples:")
active_associated.head(20)

HS10 codes with associated NAICS (from HS8): 446

Examples:


,HS10,naics,associated_naics,long_desc
907,0603190107,NaN,[111422],NaN
908,0603190109,NaN,[111422],NaN
909,0603190111,NaN,[111422],NaN
910,0603190113,NaN,[111422],NaN
912,0603190115,NaN,[111422],NaN
914,0603190117,NaN,[111422],NaN
915,0603190118,NaN,[111422],NaN
916,0603190119,NaN,[111422],NaN
919,0603190127,NaN,[111422],NaN
920,0603190129,NaN,[111422],NaN


In [42]:
active_associated.tail(20)

,HS10,naics,associated_naics,long_desc
18065,9401612015,NaN,"[337121, 337127]",NaN
18067,9401612031,NaN,"[337121, 337127]",NaN
18068,9401612035,NaN,"[337121, 337127]",NaN
18079,9401694011,NaN,"[337122, 337127]",NaN
18080,9401694015,NaN,"[337122, 337127]",NaN
18082,9401694031,NaN,"[337122, 337127]",NaN
18083,9401694035,NaN,"[337122, 337127]",NaN
18220,9403999040,NaN,"[337121, 337215]",NaN
18221,9403999045,NaN,"[337121, 337215]",NaN
18244,9404901030,NaN,[314120],NaN


In [43]:
# Check rows where naics is NaN
nan_naics = result_df[result_df['naics'].isna()]
print(f"Rows with NaN naics: {len(nan_naics)}")
print(f"Of those, rows with NaN long_desc: {nan_naics['long_desc'].isna().sum()}")
print(f"\nSample of rows with NaN naics:")
nan_naics.head(10)

Rows with NaN naics: 446
Of those, rows with NaN long_desc: 446

Sample of rows with NaN naics:


,HS10,naics,associated_naics,long_desc
907,0603190107,NaN,[111422],NaN
908,0603190109,NaN,[111422],NaN
909,0603190111,NaN,[111422],NaN
910,0603190113,NaN,[111422],NaN
912,0603190115,NaN,[111422],NaN
914,0603190117,NaN,[111422],NaN
915,0603190118,NaN,[111422],NaN
916,0603190119,NaN,[111422],NaN
919,0603190127,NaN,[111422],NaN
920,0603190129,NaN,[111422],NaN


In [44]:
my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

end_use = "naics?get=CTY_NAME,CON_VAL_YR,CAL_DUT_MO,NAICS,NAICS_LDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&time=" + "2024-12" + "&COMM_LVL=NA6" 

url = surl

r = requests.get(url) 

print(r)

naics_trade_df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
# Note that the first entry is the labels

naics_trade_df.columns = r.json()[0]

<Response [200]>


In [45]:
naics_trade_df.head()

,CTY_NAME,CON_VAL_YR,CAL_DUT_MO,NAICS,NAICS_LDESC,time,COMM_LVL
0,TOTAL FOR ALL COUNTRIES,379449116,55419,111110,SOYBEANS,2024-12,NA6
1,TOTAL FOR ALL COUNTRIES,752420965,146564,111120,OILSEEDS (EXCEPT SOYBEAN),2024-12,NA6
2,TOTAL FOR ALL COUNTRIES,631258744,490919,111130,DRY PEAS AND BEANS,2024-12,NA6
3,TOTAL FOR ALL COUNTRIES,764189160,159628,111140,WHEAT,2024-12,NA6
4,TOTAL FOR ALL COUNTRIES,254024557,741,111150,CORN,2024-12,NA6


In [46]:
my_key = "&key=34e40301bda77077e24c859c6c6c0b721ad73fc7"
# This is my key. I'm nice and I have it posted. If you will be doing more with this
# please get your own key!

end_use = "hs?get=CTY_NAME,CON_VAL_YR,I_COMMODITY,I_COMMODITY_SDESC,I_COMMODITY_LDESC"

surl = "https://api.census.gov/data/timeseries/intltrade/imports/" + end_use 

surl  = surl + my_key + "&time=" + "2024-12" + "&COMM_LVL=HS10" 

url = surl

r = requests.get(url) 

print(r)

hs10_trade_df = pd.DataFrame(r.json()[1:]) # This then converts it to a dataframe
# Note that the first entry is the labels

hs10_trade_df.columns = r.json()[0]

<Response [200]>


In [47]:
# Create a mapping dictionary from NAICS code to description
# This dictionary allows quick lookup: naics_code -> description
naics_desc_map = dict(zip(naics_trade_df['NAICS'], naics_trade_df['NAICS_LDESC']))

# Function to get NAICS descriptions based on the row's NAICS data
def get_naics_descriptions(row):
    """
    Returns NAICS description(s) based on available NAICS codes:
    - If there's a single direct NAICS match -> returns a single string (the description)
    - If there are multiple associated NAICS codes -> returns a list of descriptions
    - If there's no NAICS codes at all -> returns None
    
    The 'Description not found' fallback handles cases where a NAICS code exists 
    but isn't in our description mapping (shouldn't happen if data is complete).
    """
    
    # CASE 1: Single direct NAICS match
    # Returns: a single string with the description
    if pd.notna(row['naics']):
        naics_code = str(row['naics'])
        # Look up the description; if not found, use fallback message
        return naics_desc_map.get(naics_code, f'Description not found for NAICS {naics_code}')
    
    # CASE 2: Multiple associated NAICS codes (from HS8 match)
    # Returns: a list of descriptions, one for each NAICS code
    elif isinstance(row['associated_naics'], list) and len(row['associated_naics']) > 0:
        descriptions = []
        for naics_code in row['associated_naics']:
            naics_str = str(naics_code)
            # Look up each description; if not found, use fallback message
            desc = naics_desc_map.get(naics_str, f'Description not found for NAICS {naics_str}')
            descriptions.append(desc)
        return descriptions
    
    # CASE 3: No NAICS codes available at all
    # Returns: None
    else:
        return None

# Apply the function to create new column
result_df['naics_descriptions'] = result_df.apply(get_naics_descriptions, axis=1)

# Show results
print(f"Added NAICS descriptions column")
print(f"Rows with single description: {result_df['naics_descriptions'].apply(lambda x: isinstance(x, str)).sum():,}")
print(f"Rows with list of descriptions: {result_df['naics_descriptions'].apply(lambda x: isinstance(x, list)).sum():,}")
print(f"Rows with no descriptions: {result_df['naics_descriptions'].isna().sum():,}")
print("\nExamples (all columns):")
result_df.head(20)

Added NAICS descriptions column
Rows with single description: 18,254
Rows with list of descriptions: 446
Rows with no descriptions: 0

Examples (all columns):


,HS10,naics,associated_naics,long_desc,naics_descriptions
0,0101210010,112920,[],"HORSES, LIVE, PUREBRED BREEDING MALE",HORSES AND OTHER EQUINE
1,0101210020,112920,[],"HORSES, LIVE, PUREBRED BREEDING FEMALE",HORSES AND OTHER EQUINE
2,0101290090,112920,[],"HORSES, LIVE, NESOI",HORSES AND OTHER EQUINE
3,0101300000,112920,[],"ASSES, LIVE",HORSES AND OTHER EQUINE
4,0102210010,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING MALE, DAIRY",Description not found for NAICS 11211X
5,0102210020,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING FEMALE, DAIRY",Description not found for NAICS 11211X
6,0102210030,11211X,[],"CATTLE, PUREBRED BREEDING, MALE, LIVE, EXCEPT ...",Description not found for NAICS 11211X
7,0102210050,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING FEMALE, EXCEPT...",Description not found for NAICS 11211X
8,0102292011,11211X,[],"CATTLE, IMPORTED SPECIALLY FOR DAIRY PURPOSES,...",Description not found for NAICS 11211X
9,0102292012,11211X,[],"CATTLE, IMPORTED SPECIALLY FOR DAIRY PURPOSES,...",Description not found for NAICS 11211X


In [48]:
# Check what columns we have
print("Columns in result_df:")
print(result_df.columns.tolist())
print("\nSample with all columns:")
result_df.head(10)

Columns in result_df:
['HS10', 'naics', 'associated_naics', 'long_desc', 'naics_descriptions']

Sample with all columns:


,HS10,naics,associated_naics,long_desc,naics_descriptions
0,0101210010,112920,[],"HORSES, LIVE, PUREBRED BREEDING MALE",HORSES AND OTHER EQUINE
1,0101210020,112920,[],"HORSES, LIVE, PUREBRED BREEDING FEMALE",HORSES AND OTHER EQUINE
2,0101290090,112920,[],"HORSES, LIVE, NESOI",HORSES AND OTHER EQUINE
3,0101300000,112920,[],"ASSES, LIVE",HORSES AND OTHER EQUINE
4,0102210010,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING MALE, DAIRY",Description not found for NAICS 11211X
5,0102210020,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING FEMALE, DAIRY",Description not found for NAICS 11211X
6,0102210030,11211X,[],"CATTLE, PUREBRED BREEDING, MALE, LIVE, EXCEPT ...",Description not found for NAICS 11211X
7,0102210050,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING FEMALE, EXCEPT...",Description not found for NAICS 11211X
8,0102292011,11211X,[],"CATTLE, IMPORTED SPECIALLY FOR DAIRY PURPOSES,...",Description not found for NAICS 11211X
9,0102292012,11211X,[],"CATTLE, IMPORTED SPECIALLY FOR DAIRY PURPOSES,...",Description not found for NAICS 11211X


In [49]:
# Show instances where associated_naics is active (non-empty)
active_associated = result_df[result_df['associated_naics'].apply(len) > 0].copy()

print(f"HS10 codes with associated NAICS (from HS8): {len(active_associated):,}")
print(f"\nExamples:")
active_associated.head(20)

HS10 codes with associated NAICS (from HS8): 446

Examples:


,HS10,naics,associated_naics,long_desc,naics_descriptions
907,0603190107,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
908,0603190109,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
909,0603190111,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
910,0603190113,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
912,0603190115,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
914,0603190117,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
915,0603190118,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
916,0603190119,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
919,0603190127,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"
920,0603190129,NaN,[111422],NaN,"[FRESH FLOWERS, SEEDS AND FOLIAGE]"


In [50]:
active_associated.tail(20).iloc[0].naics_descriptions

['UPHOLSTERED HOUSEHOLD FURNITURE', 'INSTITUTIONAL FURNITURE']

In [51]:
# Fill in missing HS10 descriptions from hs10_trade_df
# Create a mapping dictionary from HS10 code to description
hs10_desc_map = dict(zip(hs10_trade_df['I_COMMODITY'], hs10_trade_df['I_COMMODITY_LDESC']))

# Count how many are missing before filling
missing_before = result_df['long_desc'].isna().sum()

# Get indices of rows with missing descriptions to show examples
missing_indices = result_df[result_df['long_desc'].isna()].index[:10].tolist()

print(f"=== BEFORE FILLING ===")
print(f"Missing HS10 descriptions: {missing_before:,}")
print(f"\nExamples of rows with missing descriptions:")
print(result_df.loc[missing_indices, ['HS10', 'long_desc', 'naics', 'associated_naics']])

# Fill in missing long_desc values
result_df['long_desc'] = result_df.apply(
    lambda row: hs10_desc_map.get(row['HS10'], row['long_desc']) if pd.isna(row['long_desc']) else row['long_desc'],
    axis=1
)

# Count how many are still missing after filling
missing_after = result_df['long_desc'].isna().sum()

print(f"\n=== AFTER FILLING ===")
print(f"Missing HS10 descriptions: {missing_after:,}")
print(f"Filled in: {missing_before - missing_after:,}")
print(f"\nSame rows after filling:")
print(result_df.loc[missing_indices, ['HS10', 'long_desc', 'naics', 'associated_naics']])
print(f"\nTotal HS10 codes with descriptions: {result_df['long_desc'].notna().sum():,} / {len(result_df):,}")

=== BEFORE FILLING ===
Missing HS10 descriptions: 446

Examples of rows with missing descriptions:
           HS10 long_desc naics associated_naics
907  0603190107       NaN   NaN         [111422]
908  0603190109       NaN   NaN         [111422]
909  0603190111       NaN   NaN         [111422]
910  0603190113       NaN   NaN         [111422]
912  0603190115       NaN   NaN         [111422]
914  0603190117       NaN   NaN         [111422]
915  0603190118       NaN   NaN         [111422]
916  0603190119       NaN   NaN         [111422]
919  0603190127       NaN   NaN         [111422]
920  0603190129       NaN   NaN         [111422]

=== AFTER FILLING ===
Missing HS10 descriptions: 0
Filled in: 446

Same rows after filling:
           HS10                                          long_desc naics  \
907  0603190107  CUT FLOWERS AND FLOWER BUDS, FRESH, OF A KIND ...   NaN   
908  0603190109  CUT FLOWERS AND FLOWER BUDS, FRESH, OF A KIND ...   NaN   
909  0603190111  CUT FLOWERS AND FLOWER B

In [52]:
result_df.head()

,HS10,naics,associated_naics,long_desc,naics_descriptions
0,0101210010,112920,[],"HORSES, LIVE, PUREBRED BREEDING MALE",HORSES AND OTHER EQUINE
1,0101210020,112920,[],"HORSES, LIVE, PUREBRED BREEDING FEMALE",HORSES AND OTHER EQUINE
2,0101290090,112920,[],"HORSES, LIVE, NESOI",HORSES AND OTHER EQUINE
3,0101300000,112920,[],"ASSES, LIVE",HORSES AND OTHER EQUINE
4,0102210010,11211X,[],"CATTLE, LIVE, PUREBRED BREEDING MALE, DAIRY",Description not found for NAICS 11211X
